# Stability classification from ERA5 data
This script derives "stable weights", that is the frequency of stable vs. non-stable conditions from freely available ERA5 data for user provided locations. We use the Earth Data Hub to rapidly download ERA5 time series signals, then process them using a selected approach to predict a stability metric time series. 

The final classification of stable weights should be made to match the methodology used to derive atmospheric conditions being used in your CFD.ML calculation.

Note: this script's execution speed greatly depends on your bandwidth - EarthDataHub access method is faster but assumes downloading a lot of data to your local machine, then slicing and filtering it to compute at a timeseries for a single grid point. When using the traditional access method, the Climate Data Store API, all this happens on the server side, involves a queueing mechanism and hence is slower, but does not load your bandtwidth or machine. That's why it might not be a good idea to execute this script when connected over a mobile hot spot with charges linked to MB transferred!

#### Creating a conda environment
We provide a conda environment to ensure that you have all the necessary dependencies.

* Open "Anaconda prompt (Miniconda 3)" from the start menu.
* Type cd "<this folder path>" and hit return. 
* Create the conda environment defined in the ```.\era5_download_environment.yaml``` file with the following command: ```conda env create -f cfdml_era5_download_environment.yaml```
* Several packages should be installed
* You should now be able to activate the new conda environment by running `conda activate cfdml_era5_download`
* Point this jupyter notebook to the newly created environment

In [10]:
import os
import pandas as pd
from datetime import datetime as dt
import script_lib.era5_signal_download_and_processing as era5
from script_lib.stable_weights import prepare_era5_stable_weights_for_api_baseline_inputs

era5_results_folder = './stable_weights_from_ERA5'

# Ensure the base folder exists
os.makedirs(era5_results_folder, exist_ok=True)

In [11]:
# default folders in which to download and create files:
results_folder = os.path.join(era5_results_folder, "stable_weights_results")
era5_cache_directory = os.path.join(era5_results_folder, "era5_time_series")

# Ensure subdirectories exist
os.makedirs(results_folder, exist_ok=True)
os.makedirs(era5_cache_directory, exist_ok=True)

## User inputs

### Define your EDH PAT key
This is a key like "edh_pat_0b05b0f1681cbef33e74f0dcdjbj1bb8d1b2df42e6fb3f5ad9171067dfe7028cbd10fec0987d00126ba9bb9cf4c7569" which you should obtain by following these steps:
* Go to the EU Destination Earth / Earth data hub website: https://earthdatahub.destine.eu/
* Sign in or create a new account (top right corner)
* Click your name then [Account Settings](https://earthdatahub.destine.eu/account-settings) (top right corner)
* Add a personal access token (PAT)
* Copy the PAT key from the tool under your personal access tokens
* Below we get the PAT from an environment variable EDH_PAT, see [here](https://mysoftware.dnv.com/download/public/renewables/windfarmer/manuals/latest/WebAPI/Introduction/apiAccessKeys.html#environment-variable-to-store-api-access-key), for guidance on how to create environment variables, or you can paste the key in directly.

In [12]:
EDH_PAT = os.environ.get("EDH_PAT")
if EDH_PAT is None:
    raise ValueError("EDH_PAT environment variable not set. Please set it to your EDH personal access token.")

In [13]:
# loading 2 years of data for each site
start = dt(2023, 1, 1, 0, 0)
end = dt(2024, 12, 31, 23, 59)

### Define the list of sites of interest by providing a list of lattitudes and longitudes
These locations should be the centre of each wind farm

In [14]:
# the script has been developed as part of a wider benchmarking exercise whereby many sites adjacent to each other shared stability weights.
# Because of this, the script is designed to treat the first word of the site name as a unique identifier.
# Sites with matching idetifiers will reference the same cached era5 data and consequently also will have the same stability weights.
# make sute to given differing names to projects far apart from one another. 

sites = {
    "site_1":(54.72420333, 6.37989345), 
    #"site_2":(54.87225313, 6.11909347)
    }

## Stability weighting methodology decision
The ERA5 stability weighting derivation method aims to emulate part of a more complex analysis with WRF. In the WRF process we categorize continuously varying atmospheric parameters into two atmospheric condition classes and we assign frequencies of occurence of those. The WRF analyses are driven by ERA5 data. We've found that using the ERA5 data directly can work very well. 

* **blend**: Using a WRF methodology based on surface temperatures to categorise stability is replicated best by a blend of the heat flux and MOL stability metrics derived from ERA5. 
* **hf**: Using WRF temperature gradients across the turbine rotor to define stability seems to match more closely to a categorisation based on ERA5 heat flux. 

The above approximations have been suggested based on benchmarking  the ERA5 stability method with past CFD jobs across several sites.

* **In regions outside North America**  a blend of the heat flux and MOL was found to replicate the methodology historically used by the CFD team when deriving stable weights.
* **Within North America**, the heat flux method has been closer to the WRF results on historic CFD projects. The CFD team have a preference for applying temperature gradient based categorisation of atmospheric stability, so heat flux is the default option here.

What is most important however, is that the methodology used for deriving stable weights matches best with the methodology used to define the atmospheric conditions. The atmospheric conditions are defined by splitting a time series of vertical profiles into 2 bins and averaging. **That splitting should be roughly consistent with that used to define stable/non-stable frequency weighting.**

### Define the methodologies you may like to use to derive stable weights. 
This will define the signals we will download for your sites
Uncomment the methodologies you wish to explore

In [ ]:
# reccomended options: "blend", "hf"
era5_method = "blend" 
number_of_direction_sectors = 12

## Download ERA5 time series data

In [16]:
signal_names_to_download = ['u10', 'v10']
# Heat flux
if era5_method == "hf":
    signal_names_to_download = signal_names_to_download + era5.get_era5_signal_names_for_heat_flux_method()
elif era5_method == "blend":
    signal_names_to_download = signal_names_to_download + era5.get_era5_signal_names_for_blended_mol_hf_method()
signal_names_to_download

['u10', 'v10', 'sshf']

In [18]:
# do the downloads
for site, coords in sites.items():
    lat = coords[0]
    lon = coords[1]
    print("Loading site: ", site)
    xarray_ds = era5.get_era5_model_level_dataset(EDH_PAT)
    df = era5.get_era5_subset_as_df(xarray_ds, signal_names_to_download, lat, lon, start, end)
    df.to_csv(f"./{era5_cache_directory}/era5_{site}.csv", index=True)

Loading site:  site_1
loading time slice 2023-01-01T00:00:00.000000 to 2023-01-31T00:00:00.000000000
	loading variables: ['sshf', 'u10', 'v10']
		Size of the slice: 14.11328125 kB
		Execution time: 94.39 seconds
loading time slice 2023-01-31T00:00:00.000000000 to 2024-01-31T00:00:00.000000000
	loading variables: ['sshf', 'u10', 'v10']
		Size of the slice: 171.14453125 kB
		Execution time: 193.16 seconds
loading time slice 2024-01-31T00:00:00.000000000 to 2024-12-31T23:59:00.000000
	loading variables: ['sshf', 'u10', 'v10']
		Size of the slice: 157.53125 kB
		Execution time: 160.03 seconds


## Generate stable weight data from the downloaded ERA5 time series

In [19]:
# process the freshly downloaded data and add the era5_stable_weights.txt files where needed  
for project in sites.keys():
    exported_weights_file = prepare_era5_stable_weights_for_api_baseline_inputs(project, results_folder, era5_cache_directory, era5_method, True, number_of_direction_sectors)
    stable_weights_df = pd.read_csv(exported_weights_file, sep='\t', index_col=0)
    print(stable_weights_df)

ERA5 stable weights for project site_1 written to ./02.stable_weights_from_ERA5\stable_weights_results\site_1\era5_stable_weights.txt
            stable_weight
bin_centre               
0.0              0.168806
30.0             0.244819
60.0             0.309904
90.0             0.442520
120.0            0.480903
150.0            0.418406
180.0            0.485352
210.0            0.512500
240.0            0.442538
270.0            0.280702
300.0            0.113290
330.0            0.066372
